# Experiment 6: Results Visualization
## Load saved JSON results and generate comprehensive plots

In [ ]:
import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f8f9fa",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

# Try multiple locations: relative to notebook (results/) or from repo root (experiment_6/results/)
RESULTS_DIR = "results"
if not os.path.isdir(RESULTS_DIR):
    RESULTS_DIR = "experiment_6/results"

def load_json(path):
    with open(path) as f:
        return json.load(f)

# Load both result files (if available)
data = None
data_2k = None

small_path = os.path.join(RESULTS_DIR, "final_results.json")
large_path = os.path.join(RESULTS_DIR, "final_results_2000.json")

if os.path.exists(small_path):
    data = load_json(small_path)
    print(f"Loaded final_results.json  — {data.get('finetune_steps', '?'):,} finetune steps")
    print(f"  Pretrain epochs: {len(data.get('pretrain_history', []))}")
    print(f"  Training metrics: {len(data.get('training_metrics', []))} snapshots")
else:
    print("final_results.json not found")

if os.path.exists(large_path):
    data_2k = load_json(large_path)
    print(f"\nLoaded final_results_2000.json  — {data_2k.get('num_symbols', '?'):,} symbols")
    print(f"  Pretrain epochs: {len(data_2k.get('pretrain_history', []))}")
    print(f"  Training progress: {len(data_2k.get('training_progress', []))} snapshots")
else:
    print("\nfinal_results_2000.json not found")

## 1. Pretraining: Loss & Accuracy

In [ ]:
def plot_pretraining(pretrain_hist, title_prefix=""):
    if not pretrain_hist:
        print("No pretrain data")
        return
    epochs = [h["epoch"] for h in pretrain_hist]
    losses = [h["loss"] for h in pretrain_hist]
    accs   = [h["accuracy"] * 100 for h in pretrain_hist]

    fig, ax1 = plt.subplots(figsize=(12, 5))
    ax2 = ax1.twinx()

    ax1.plot(epochs, losses, "b-o", markersize=5, linewidth=2, label="Loss")
    ax2.plot(epochs, accs, "g-s", markersize=5, linewidth=2, label="Accuracy %")

    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss", color="blue")
    ax2.set_ylabel("Accuracy (%)", color="green")
    ax1.set_title(f"{title_prefix}Expert Behavioral Cloning — Pretraining")
    ax1.legend(loc="upper left")
    ax2.legend(loc="upper right")
    ax1.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

if data and data.get("pretrain_history"):
    plot_pretraining(data["pretrain_history"], title_prefix="(Small) ")
if data_2k and data_2k.get("pretrain_history"):
    plot_pretraining(data_2k["pretrain_history"], title_prefix="(2000-stock) ")

## 2. Training Progress: Validation Metrics vs Steps

In [ ]:
def plot_training_progress(metrics, title_prefix=""):
    if not metrics:
        print("No training metrics")
        return
    steps   = [m.get("step", 0) for m in metrics]
    rets    = [m.get("return_pct", 0) for m in metrics]
    sharpes = [m.get("sharpe", 0) for m in metrics]
    max_dds = [m.get("max_dd", m.get("max_dd_pct", 0)) for m in metrics]
    n_trades= [m.get("n_trades", m.get("num_trades", 0)) for m in metrics]

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    # Return %
    ax = axes[0, 0]
    ax.plot(steps, rets, "b-", linewidth=2)
    ax.fill_between(steps, 0, rets, where=(np.array(rets)>=0), alpha=0.15, color="green")
    ax.fill_between(steps, 0, rets, where=(np.array(rets)<0),  alpha=0.15, color="red")
    ax.axhline(y=0, color="red", ls="--", alpha=0.5)
    ax.set_title("Validation Return %")
    ax.set_ylabel("Return (%)")
    ax.grid(alpha=0.2)

    # Sharpe
    ax = axes[0, 1]
    ax.plot(steps, sharpes, "g-", linewidth=2)
    ax.fill_between(steps, 0, sharpes, where=(np.array(sharpes)>=0), alpha=0.15, color="green")
    ax.fill_between(steps, 0, sharpes, where=(np.array(sharpes)<0),  alpha=0.15, color="red")
    ax.axhline(y=0, color="red", ls="--", alpha=0.5)
    ax.set_title("Validation Sharpe Ratio")
    ax.set_ylabel("Sharpe")
    ax.grid(alpha=0.2)

    # Max Drawdown
    ax = axes[1, 0]
    ax.plot(steps, max_dds, "r-", linewidth=2)
    ax.fill_between(steps, 0, max_dds, alpha=0.15, color="red")
    ax.set_title("Validation Max Drawdown %")
    ax.set_xlabel("Timesteps")
    ax.set_ylabel("Drawdown (%)")
    ax.invert_yaxis()
    ax.grid(alpha=0.2)

    # Number of Trades
    ax = axes[1, 1]
    ax.bar(steps, n_trades, width=max(500, (steps[-1]-steps[0])//len(steps)*0.6), color="steelblue", alpha=0.7, edgecolor="black", linewidth=0.3)
    ax.set_title("Number of Trades per Eval")
    ax.set_xlabel("Timesteps")
    ax.set_ylabel("# Trades")
    ax.grid(alpha=0.2)

    fig.suptitle(f"{title_prefix}PPO Fine-Tuning Progress", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

if data and data.get("training_metrics"):
    plot_training_progress(data["training_metrics"], title_prefix="(Small) ")
if data_2k and data_2k.get("training_progress"):
    plot_training_progress(data_2k["training_progress"], title_prefix="(2000-stock) ")

## 3. Final Results Summary

In [ ]:
def plot_final_summary(d, title_prefix=""):
    """Plot bar chart comparing val vs test metrics."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # --- Left: Key metrics table ---
    ax = axes[0]
    ax.axis("off")

    # Gather metrics (handles both JSON formats)
    v_ret  = d.get("val_return_pct", d.get("val_metrics", {}).get("return_pct", 0))
    t_ret  = d.get("test_return_pct", d.get("test_metrics", {}).get("return_pct", 0))
    t_sh   = d.get("test_sharpe", d.get("test_metrics", {}).get("sharpe", 0))
    t_dd   = d.get("test_max_dd", d.get("test_metrics", {}).get("max_dd_pct", 0))
    t_time = d.get("train_time_seconds", 0) / 60
    device = d.get("device", "?")
    p_steps= d.get("pretrain_steps", 0)
    f_steps= d.get("finetune_steps", 0)
    n_sym  = d.get("num_symbols", "N/A")
    n_tr   = d.get("n_test_trades", d.get("test_metrics", {}).get("num_trades", 0))
    wr     = d.get("test_metrics", {}).get("win_rate", 0)
    awin   = d.get("test_metrics", {}).get("avg_win_pct", 0)
    aloss  = d.get("test_metrics", {}).get("avg_loss_pct", 0)

    rows = [
        ["Metric", "Value"],
        ["Symbols", str(n_sym)],
        ["Pretrain Steps", f"{p_steps:,}"],
        ["Finetune Steps", f"{f_steps:,}"],
        ["Train Time", f"{t_time:.1f} min"],
        ["Device", device],
        ["", ""],
        ["Val Return %", f"{v_ret:+.2f}%"],
        ["Test Return %", f"{t_ret:+.2f}%"],
        ["Test Sharpe", f"{t_sh:.4f}"],
        ["Test Max DD %", f"{t_dd:.2f}%"],
        ["Test Trades", str(n_tr)],
    ]
    if wr:
        rows += [
            ["Win Rate", f"{wr*100:.1f}%"],
            ["Avg Win %", f"{awin:.4f}%"],
            ["Avg Loss %", f"{aloss:.4f}%"],
        ]

    col_labels = rows[0]
    cell_text  = rows[1:]
    table = ax.table(cellText=cell_text, colLabels=col_labels,
                     cellLoc="center", loc="center",
                     colWidths=[0.4, 0.4])
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_facecolor("#4472C4")
            cell.set_text_props(color="white", fontweight="bold")
        elif row % 2 == 0:
            cell.set_facecolor("#e8edf7")
    ax.set_title(f"{title_prefix}Final Results Summary", fontweight="bold")

    # --- Right: Bar chart comparison ---
    ax = axes[1]
    labels = ["Val Return %", "Test Return %", "Test Sharpe", "Max DD %"]
    values = [v_ret, t_ret, t_sh, t_dd]
    colors = ["#5b9bd5", "#4472C4", "#70ad47", "#ff6b6b"]
    bars = ax.barh(labels, values, color=colors, alpha=0.85, edgecolor="black", linewidth=0.5)
    ax.axvline(x=0, color="black", lw=0.8)
    ax.set_title("Key Metrics")
    for bar, v in zip(bars, values):
        ax.text(bar.get_width() + (0.1 if v>=0 else -0.5),
                bar.get_y() + bar.get_height()/2,
                f"{v:+.2f}", va="center", fontsize=10, fontweight="bold")
    ax.grid(alpha=0.2)

    plt.tight_layout()
    plt.show()

if data:
    plot_final_summary(data, title_prefix="(Small) ")
if data_2k:
    plot_final_summary(data_2k, title_prefix="(2000-stock) ")

## 4. Hyperparameters Overview

In [ ]:
def plot_hyperparams(d, title_prefix=""):
    bp = d.get("best_params", {})
    if not bp:
        print("No hyperparams data")
        return
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.axis("off")
    param_rows = [["Parameter", "Value"]]
    for k, v in bp.items():
        param_rows.append([k, str(v)])
    tab = ax.table(cellText=param_rows[1:], colLabels=param_rows[0],
                   cellLoc="center", loc="center",
                   colWidths=[0.35, 0.55])
    tab.auto_set_font_size(False)
    tab.set_fontsize(10)
    for (row, col), cell in tab.get_celld().items():
        if row == 0:
            cell.set_facecolor("#4472C4")
            cell.set_text_props(color="white", fontweight="bold")
        elif row % 2 == 0:
            cell.set_facecolor("#e8edf7")
    ax.set_title(f"{title_prefix}Best Hyperparameters", fontweight="bold")
    plt.tight_layout()
    plt.show()

if data and data.get("best_params"):
    plot_hyperparams(data, title_prefix="(Small) ")
if data_2k and data_2k.get("best_params"):
    plot_hyperparams(data_2k, title_prefix="(2000-stock) ")

## 5. Combined Dashboard (all in one figure)

In [ ]:
def create_dashboard(d, pretrain_hist, train_metrics, title_label="", save_path=None):
    """9-panel comprehensive dashboard (matches train.py output style)."""
    fig, axes = plt.subplots(3, 3, figsize=(22, 16))

    # ---- Row 0 ----
    # [0,0] Pretrain Loss & Accuracy
    ax = axes[0, 0]
    if pretrain_hist:
        epochs = [h["epoch"] for h in pretrain_hist]
        losses = [h["loss"] for h in pretrain_hist]
        accs   = [h["accuracy"] * 100 for h in pretrain_hist]
        ax2 = ax.twinx()
        ax.plot(epochs, losses, "b-o", markersize=3, lw=1.5, label="Loss")
        ax2.plot(epochs, accs, "g-s", markersize=3, lw=1.5, label="Accuracy %")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
        ax2.set_ylabel("Accuracy (%)")
        ax.set_title("Pretraining: Loss & Accuracy")
        ax.legend(loc="upper left"); ax2.legend(loc="upper right")
    else:
        ax.text(0.5, 0.5, "No pretrain data", ha="center", va="center")
        ax.set_title("Pretraining")
    ax.grid(alpha=0.2)

    # [0,1] Return vs Steps
    ax = axes[0, 1]
    if train_metrics:
        steps = [m.get("step", 0) for m in train_metrics]
        rets  = [m.get("return_pct", 0) for m in train_metrics]
        ax.plot(steps, rets, "b-", linewidth=2)
        ax.axhline(y=0, color="r", ls="--", alpha=0.5)
        ax.fill_between(steps, 0, rets, where=(np.array(rets)>=0), alpha=0.15, color="green")
        ax.fill_between(steps, 0, rets, where=(np.array(rets)<0), alpha=0.15, color="red")
        ax.set_title("Val Return % vs Steps")
        ax.set_ylabel("Return (%)")
    else:
        ax.text(0.5, 0.5, "No training data", ha="center", va="center")
    ax.grid(alpha=0.2)

    # [0,2] Sharpe vs Steps
    ax = axes[0, 2]
    if train_metrics:
        sharpes = [m.get("sharpe", 0) for m in train_metrics]
        ax.plot(steps, sharpes, "g-", linewidth=2)
        ax.axhline(y=0, color="r", ls="--", alpha=0.5)
        ax.set_title("Val Sharpe vs Steps")
        ax.set_ylabel("Sharpe")
    else:
        ax.text(0.5, 0.5, "No training data", ha="center", va="center")
    ax.grid(alpha=0.2)

    # ---- Row 1 ----
    # [1,0] DD vs Steps
    ax = axes[1, 0]
    if train_metrics:
        max_dds = [m.get("max_dd", m.get("max_dd_pct", 0)) for m in train_metrics]
        ax.plot(steps, max_dds, "r-", linewidth=2)
        ax.fill_between(steps, 0, max_dds, alpha=0.15, color="red")
        ax.set_title("Val Max DD % vs Steps")
        ax.set_xlabel("Timesteps"); ax.set_ylabel("Drawdown (%)")
        ax.invert_yaxis()
    else:
        ax.text(0.5, 0.5, "No training data", ha="center", va="center")
    ax.grid(alpha=0.2)

    # [1,1] Trade Count vs Steps
    ax = axes[1, 1]
    if train_metrics:
        n_trades = [m.get("n_trades", m.get("num_trades", 0)) for m in train_metrics]
        ax.bar(steps, n_trades, width=max(500, (steps[-1]-steps[0])//len(steps)*0.6),
               color="steelblue", alpha=0.7, edgecolor="black", lw=0.3)
        ax.set_title("# Trades per Eval")
        ax.set_xlabel("Timesteps"); ax.set_ylabel("# Trades")
    else:
        ax.text(0.5, 0.5, "No training data", ha="center", va="center")
    ax.grid(alpha=0.2)

    # [1,2] Win Rate vs Steps (2000-stock data only)
    ax = axes[1, 2]
    if train_metrics and "win_rate" in train_metrics[0]:
        win_rates = [m.get("win_rate", 0) * 100 for m in train_metrics]
        ax.plot(steps, win_rates, "purple", linewidth=2, marker="o", markersize=3)
        ax.set_title("Val Win Rate % vs Steps")
        ax.set_xlabel("Timesteps"); ax.set_ylabel("Win Rate (%)")
    else:
        ax.text(0.5, 0.5, "Win rate data\nnot available", ha="center", va="center")
        ax.set_title("Win Rate")
    ax.grid(alpha=0.2)

    # ---- Row 2: Final summary numbers ----
    v_ret = d.get("val_return_pct", d.get("val_metrics", {}).get("return_pct", 0))
    t_ret = d.get("test_return_pct", d.get("test_metrics", {}).get("return_pct", 0))
    t_sh  = d.get("test_sharpe", d.get("test_metrics", {}).get("sharpe", 0))
    t_dd  = d.get("test_max_dd", d.get("test_metrics", {}).get("max_dd_pct", 0))
    t_tr  = d.get("n_test_trades", d.get("test_metrics", {}).get("num_trades", 0))
    t_time= d.get("train_time_seconds", 0) / 60
    p_st  = d.get("pretrain_steps", 0)
    f_st  = d.get("finetune_steps", 0)
    n_sym = d.get("num_symbols", "N/A")

    # [2,0] Text summary
    ax = axes[2, 0]
    ax.axis("off")
    lines = [
        f"Symbols: {n_sym}",
        f"Pretrain: {p_st:,} steps | Finetune: {f_st:,} steps",
        f"Train Time: {t_time:.1f} min",
        "",
        f"Val Return:   {v_ret:+.2f}%",
        f"Test Return:   {t_ret:+.2f}%",
        f"Test Sharpe:   {t_sh:.4f}",
        f"Test Max DD:   {t_dd:.2f}%",
        f"Test Trades:   {t_tr}",
    ]
    for i, line in enumerate(lines):
        y = 1 - i * 0.09
        weight = "bold" if i in (0, 4) else "normal"
        ax.text(0.1, y, line, transform=ax.transAxes, fontsize=11, fontweight=weight,
                family="monospace")
    ax.set_title("Final Summary", fontweight="bold")

    # [2,1] Bar chart
    ax = axes[2, 1]
    labels = ["Val Ret %", "Test Ret %", "Sharpe", "Max DD %"]
    vals   = [v_ret, t_ret, t_sh, t_dd]
    colors = ["#5b9bd5", "#4472C4", "#70ad47", "#ff6b6b"]
    ax.barh(labels, vals, color=colors, alpha=0.85, edgecolor="black", lw=0.5)
    ax.axvline(x=0, color="black", lw=0.8)
    ax.set_title("Key Metrics")
    ax.grid(alpha=0.2)

    # [2,2] Hyperparams summary
    ax = axes[2, 2]
    ax.axis("off")
    bp = d.get("best_params", {})
    if bp:
        hp_lines = [f"{k}: {v}" for k, v in bp.items()]
    else:
        hp_lines = ["No hyperparams"]
    for i, line in enumerate(hp_lines):
        ax.text(0.05, 1 - i * 0.08, line, transform=ax.transAxes,
                fontsize=9, family="monospace")
    ax.set_title("Best Hyperparameters", fontweight="bold")

    fig.suptitle(f"Experiment 6: {title_label}Comprehensive Report",
                 fontsize=16, fontweight="bold")
    plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved dashboard to {save_path}")
    plt.show()

# Generate dashboards
if data:
    create_dashboard(data, data.get("pretrain_history", []),
                     data.get("training_metrics", []),
                     title_label="(Small) ",
                     save_path=os.path.join(RESULTS_DIR, "dashboard_report.png"))
if data_2k:
    create_dashboard(data_2k, data_2k.get("pretrain_history", []),
                     data_2k.get("training_progress", []),
                     title_label="(2000-Stock) ",
                     save_path=os.path.join(RESULTS_DIR, "dashboard_2000.png"))

print("\nAll plots generated.")